<a href="https://colab.research.google.com/github/akmalfer/Disertasi-Related/blob/main/Arch_Check_Binary_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from PIL import Image
from scipy import ndimage
import os
DATADIR = "/content/drive/My Drive/DLData"    # Data untuk Deep Learning
dataset_path  = DATADIR + "/TaMuDecay"

In [ ]:
os.listdir(dataset_path)

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, LeakyReLU, Add, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
def load_data(file_path, label):
    # Read the file
    data = pd.read_csv(file_path, sep=r'\s+', header=None, dtype=str)  # Read as strings

    # Remove trailing commas and convert to float
    data = data.applymap(lambda x: float(x.replace(',', '')) if isinstance(x, str) else x)

    # Assign column names
    data.columns = ["mu_px", "mu_py", "mu_pz", "e_px", "e_py", "e_pz", "nu_px", "nu_py", "mvis"]

    # Add label column
    data["Label"] = label
    return data

In [ ]:
# ===============================
# 1️⃣ Load data function
# ===============================
def load_data(file_path, label):

    # Read as string first (to remove commas safely)
    data = pd.read_csv(file_path, sep=r'\s+', header=None, dtype=str)

    # Remove trailing commas and convert to float
    data = data.applymap(lambda x: float(x.replace(',', '')) if isinstance(x, str) else x)

    # Assign column names
    data.columns = [
        "mu_px", "mu_py", "mu_pz",
        "e_px",  "e_py",  "e_pz",
        "nu_px", "nu_py",
        "mvis"
    ]

    # Add label
    data["Label"] = label

    return data


# ===============================
# 2️⃣ Recompute visible mass
# ===============================
def recompute_mvis(df):

    m_mu = 0.105      # GeV
    m_e  = 0.000511   # GeV

    E_mu = np.sqrt(
        df["mu_px"]**2 +
        df["mu_py"]**2 +
        df["mu_pz"]**2 +
        m_mu**2
    )

    E_e = np.sqrt(
        df["e_px"]**2 +
        df["e_py"]**2 +
        df["e_pz"]**2 +
        m_e**2
    )

    px_vis = df["mu_px"] + df["e_px"]
    py_vis = df["mu_py"] + df["e_py"]
    pz_vis = df["mu_pz"] + df["e_pz"]

    E_vis = E_mu + E_e

    mvis_sq = E_vis**2 - (px_vis**2 + py_vis**2 + pz_vis**2)

    # protect against negative numerical precision
    mvis_sq = np.maximum(mvis_sq, 0)

    df["mvis"] = np.sqrt(mvis_sq)

    return df


# ===============================
# 3️⃣ Recompute collinear mass
# ===============================
def recompute_mcol(df):

    # Electron transverse momentum
    pt_e = np.sqrt(df["e_px"]**2 + df["e_py"]**2)

    # Protect against pt_e = 0
    pt_e_safe = np.where(pt_e == 0, 1e-12, pt_e)

    # Estimate neutrino pT projected onto electron
    pt_nu_est = (
        df["e_px"] * df["nu_px"] +
        df["e_py"] * df["nu_py"]
    ) / pt_e_safe

    # xvis
    denom = pt_e + pt_nu_est
    denom_safe = np.where(np.abs(denom) < 1e-12, 1e-12, denom)

    xvis = pt_e / denom_safe

    # Only valid collinear solutions
    valid = (xvis > 0) & (xvis < 1)

    mcol = np.zeros(len(df))
    mcol[valid] = df.loc[valid, "mvis"] / np.sqrt(xvis[valid])

    df["xvis"] = xvis
    df["mcol"] = mcol

    return df


# ===============================
# 4️⃣ Full pipeline
# ===============================

# Load
data_A = load_data(
    f"{dataset_path}/combined_each22965_traindata0j1mruwmvisv1.txt",
    label=0
)

# Drop old mvis
data_A = data_A.drop(columns=["mvis"])

# Recompute mvis
data_A = recompute_mvis(data_A)

# Compute collinear mass
data_A = recompute_mcol(data_A)

# Check result
print(data_A.head())
print("Columns:", data_A.columns)
print("Total events:", len(data_A))


In [ ]:
# Reload fresh copy to preserve original mvis
data_compare = load_data(
    f"{dataset_path}/combined_each22965_traindata0j1mruwmvisv1.txt",
    label=0
)

# Rename original column
data_compare = data_compare.rename(columns={"mvis": "mvis_old"})

data_compare = recompute_mvis(data_compare)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.scatter(data_compare["mvis_old"],
            data_compare["mvis"],
            s=5)

plt.xlabel("Old mvis")
plt.ylabel("Recomputed mvis")
plt.title("Old vs Recomputed mvis")

# Add diagonal reference line
min_val = min(data_compare["mvis_old"].min(), data_compare["mvis"].min())
max_val = max(data_compare["mvis_old"].max(), data_compare["mvis"].max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.show()


In [ ]:
print(data_A.head())

In [ ]:
with open(f"{dataset_path}/filtered_150to500_taulep_signal0j_wmvis_v2.csv") as f:
    for i, line in enumerate(f, start=1):
        fields = line.strip().split(',')
        if len(fields) != 10:
            print(f"Line {i}: {len(fields)} fields")

In [ ]:
# Load CSV and keep all columns
data = pd.read_csv(f"{dataset_path}/filtered_150to500_taulep_signal0j_wmvis_v2.csv", sep=",", header=None, low_memory=False)

# Drop header row only, keep all columns including msc
data = data.iloc[1:, :]

# Convert all columns to float
data = data.astype(float)

# Rename columns, assuming first column is 'msc'
data.columns = ["msc", "mu_px", "mu_py", "mu_pz", "e_px", "e_py", "e_pz", "nu_px", "nu_py", "mvis"]

# Filter msc range
data = data[(data["msc"] >= 200) & (data["msc"] <= 450)]

# Bin msc into categories to enable stratified sampling


num_bins = 50
data["msc_bin"] = pd.cut(data["msc"], bins=num_bins, labels=False)
total_samples = 45930
samples_per_bin = total_samples // num_bins  # integer division

# Sample uniformly across bins
data_B = (
    data.groupby("msc_bin", group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), samples_per_bin), random_state=42))
)
# Optional: if total is slightly under 45930 (due to small bins), fix it
if len(data_B) > total_samples:
    data_B = data_B.sample(n=total_samples, random_state=42)

In [ ]:
# Drop helper bin column
data_B = data_B.drop(columns=["msc_bin"])

# Add label
data_B["Label"] = 1

In [ ]:
# Rename original mvis
data_B = data_B.rename(columns={"mvis": "mvis_old"})


In [ ]:
def recompute_mvis(df):

    m_mu = 0.105
    m_e  = 0.000511

    E_mu = np.sqrt(
        df["mu_px"]**2 +
        df["mu_py"]**2 +
        df["mu_pz"]**2 +
        m_mu**2
    )

    E_e = np.sqrt(
        df["e_px"]**2 +
        df["e_py"]**2 +
        df["e_pz"]**2 +
        m_e**2
    )

    px_vis = df["mu_px"] + df["e_px"]
    py_vis = df["mu_py"] + df["e_py"]
    pz_vis = df["mu_pz"] + df["e_pz"]

    E_vis = E_mu + E_e

    mvis_sq = E_vis**2 - (px_vis**2 + py_vis**2 + pz_vis**2)
    mvis_sq = np.maximum(mvis_sq, 0)

    df["mvis"] = np.sqrt(mvis_sq)

    return df


data_B = recompute_mvis(data_B)


def recompute_mcol(df):

    pt_e = np.sqrt(df["e_px"]**2 + df["e_py"]**2)
    pt_e_safe = np.where(pt_e == 0, 1e-12, pt_e)

    pt_nu_est = (
        df["e_px"] * df["nu_px"] +
        df["e_py"] * df["nu_py"]
    ) / pt_e_safe

    denom = pt_e + pt_nu_est
    denom_safe = np.where(np.abs(denom) < 1e-12, 1e-12, denom)

    xvis = pt_e / denom_safe

    valid = (xvis > 0) & (xvis < 1)

    mcol = np.zeros(len(df))
    mcol[valid] = df.loc[valid, "mvis"] / np.sqrt(xvis[valid])

    df["xvis"] = xvis
    df["mcol"] = mcol

    return df


data_B = recompute_mcol(data_B)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.scatter(data_B["mvis_old"], data_B["mvis"], s=5)

plt.xlabel("Old mvis")
plt.ylabel("Recomputed mvis")
plt.title("Old vs Recomputed mvis (data_B)")

min_val = min(data_B["mvis_old"].min(), data_B["mvis"].min())
max_val = max(data_B["mvis_old"].max(), data_B["mvis"].max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.show()


In [ ]:
data_B_msc = data_B

# Drop columns not needed for training
data_B = data_B.drop(columns=["msc", "mvis_old"])

In [ ]:
final_columns = [
    "mu_px", "mu_py", "mu_pz",
    "e_px",  "e_py",  "e_pz",
    "nu_px", "nu_py",
    "mvis",
    "mcol",
    "Label"
]

data_A = data_A[final_columns]
data_B = data_B[final_columns]

In [ ]:
print("data_A shape:", data_A.shape)
print("data_B shape:", data_B.shape)

print("\ndata_A columns:\n", data_A.columns)
print("\ndata_B columns:\n", data_B.columns)

In [ ]:
feature_cols = [
    "mu_px", "mu_py", "mu_pz",
    "e_px", "e_py", "e_pz",
    "nu_px", "nu_py",
    "mvis",
    "mcol"
]

In [ ]:
# Combine datasets
data = pd.concat([data_A, data_B], ignore_index=True)

# Ensure Label is last (optional, not required for ML)
other_columns = [col for col in data.columns if col != 'Label']
data = data[other_columns + ['Label']]

# ----------------------------
# SPLIT DATAFRAME FIRST
# ----------------------------

df_train, df_test = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["Label"]   # keeps signal/background balanced
)

# ----------------------------
# Extract features and labels
# ----------------------------

X_train = df_train[feature_cols].values
y_train = df_train["Label"].values

X_test  = df_test[feature_cols].values
y_test  = df_test["Label"].values

# ----------------------------
# Scale
# ----------------------------

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
# Define the model
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # Input layer with the correct number of features

    # First Dense Block
    Dense(128),  # More neurons for deeper learning
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),

    # Second Dense Block
    Dense(64),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),

    # Third Dense Block
    Dense(64),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),

    #Check with LeakyReLU between, without BN
    Dense(32),
    LeakyReLU(alpha=0.1),

    # Fourth Dense Block
    Dense(32),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),

    # Output Layer
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),  # Adaptive optimizer
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Learning rate scheduler to dynamically adjust learning rate
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)


model.summary()

In [ ]:
# Define EarlyStopping callback

early_stopping = EarlyStopping(
    monitor='val_accuracy',   # Monitor validation accuracy
    patience=100,          # Stop if no improvement after 50 epochs
    min_delta=0.001,
    restore_best_weights=True  # Restore the best model weights
)

In [ ]:
# Train the model with EarlyStopping
history = model.fit(
    X_train, y_train,
    epochs=500,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping]  # Add the callback here
)

print("Accuracy" , model.evaluate(X_test, y_test)[1])

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_score = model.predict(X_test, batch_size=4096).ravel()

fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Generate predictions on test set
predict = model.predict(X_test)
predict = np.round(predict)  # Convert probabilities to binary predictions (0 or 1), change to 0.3 (another values other than 0.5)

# Get actual labels
actual = np.array(y_test)

# Define labels for the confusion matrix (assuming 0 and 1 are the class labels)
labels = [0, 1]

# Compute confusion matrix
conf_mat_orig = confusion_matrix(actual, predict)

# Visualize confusion matrix
displ = ConfusionMatrixDisplay(confusion_matrix=conf_mat_orig, display_labels=labels)
displ.plot()

In [ ]:
model.save(dataset_path + "/backsignsepw1mdatav_arch_check_model.keras") # add .keras extension to the file path

In [ ]:
model = keras.models.load_model(dataset_path + "/backsignsepw1mdatav_arch_check_model.keras")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import rcParams

# ── Publication style ────────────────────────────────────────────────
rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":        13,
    "axes.titlesize":   14,
    "axes.labelsize":   13,
    "xtick.labelsize":  11,
    "ytick.labelsize":  11,
    "legend.fontsize":  11,
    "axes.linewidth":   1.2,
    "xtick.direction":  "in",
    "ytick.direction":  "in",
    "xtick.top":        True,
    "ytick.right":      True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "figure.dpi":       150,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
})

CUT_VAL = 0.5
window  = 5
feature_cols_no_msc = feature_cols

# ── Scores ───────────────────────────────────────────────────────────
X_bkg            = data_A[feature_cols_no_msc].values
background_scores = model.predict(scaler.transform(X_bkg)).flatten()

sig_200 = data_B[(data_B_msc["msc"] >= 200-window) & (data_B_msc["msc"] < 200+window)]
signal_200_scores = model.predict(scaler.transform(sig_200[feature_cols_no_msc].values)).flatten()

sig_450 = data_B[(data_B_msc["msc"] >= 450-window) & (data_B_msc["msc"] < 450+window)]
signal_450_scores = model.predict(scaler.transform(sig_450[feature_cols_no_msc].values)).flatten()

# ── Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4.5))

BINS = 50
hist_kw = dict(bins=BINS, density=True, histtype="step", linewidth=1.4)

ax.hist(background_scores,  **hist_kw, color="#0057B8", alpha=0.35, edgecolor="#0057B8", label="Background")
ax.hist(signal_200_scores,  **hist_kw, color="#D00000", alpha=0.45, edgecolor="#D00000", label=r"Signal ($m_H$ = 200 GeV)")
ax.hist(signal_450_scores,  **hist_kw, color="#008B45", alpha=0.45, edgecolor="#008B45", label=r"Signal ($m_H$ = 450 GeV)")

# ── Cut line ─────────────────────────────────────────────────────────
ax.axvline(CUT_VAL, color="crimson", linestyle="--", linewidth=1.4, label=f"NN Score = {CUT_VAL}")
ax.text(CUT_VAL + 0.01, ax.get_ylim()[1] * 0.92, f"NN Score = {CUT_VAL}",
        color="crimson", fontsize=10, va="top")

# ── Labels & legend ──────────────────────────────────────────────────
ax.set_xlabel("NN Classifier Score", labelpad=8)
ax.set_ylabel("Normalised Density",  labelpad=8)
#ax.set_title("Classifier Score Distribution", pad=10)
ax.set_xlim(0, 1)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

legend = ax.legend(frameon=True, framealpha=0.9, edgecolor="gray",
                   loc="upper center", ncol=1)

# ── Watermark (optional — remove for blind review) ───────────────────
# ax.text(0.98, 0.98, "Preliminary", transform=ax.transAxes,
#         fontsize=10, color="gray", alpha=0.5,
#         ha="right", va="top", style="italic")

plt.tight_layout()
plt.savefig("classifier_score.pdf")   # vector PDF for journal
plt.savefig("classifier_score.png")   # raster PNG for preview
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── Publication style ────────────────────────────────────────────────
rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":        13,
    "axes.titlesize":   14,
    "axes.labelsize":   13,
    "xtick.labelsize":  11,
    "ytick.labelsize":  11,
    "legend.fontsize":  11,
    "axes.linewidth":   1.2,
    "xtick.direction":  "in",
    "ytick.direction":  "in",
    "xtick.top":        True,
    "ytick.right":      True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "figure.dpi":       150,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
})

# ── Build (y_true, y_score) pairs from existing score arrays ─────────
# Signal = 1, Background = 0
def make_roc(sig_scores, bkg_scores):
    y_score = np.concatenate([bkg_scores, sig_scores])
    y_true  = np.concatenate([np.zeros(len(bkg_scores)),
                               np.ones(len(sig_scores))])
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return fpr, tpr, auc(fpr, tpr)

fpr_200, tpr_200, auc_200 = make_roc(signal_200_scores, background_scores)
fpr_450, tpr_450, auc_450 = make_roc(signal_450_scores, background_scores)

# ── Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4.5))

ax.plot(fpr_200, tpr_200, color="#4C72B0", lw=2,   # soft blue
        label=rf"$m_H = 200$ GeV  (AUC $= {auc_200:.3f}$)")
ax.plot(fpr_450, tpr_450, color="#C44E52", lw=2,   # muted red
        label=rf"$m_H = 450$ GeV  (AUC $= {auc_450:.3f}$)")
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier")

# ── Axes ─────────────────────────────────────────────────────────────
ax.set_xlabel("False Positive Rate", labelpad=8)
ax.set_ylabel("True Positive Rate",  labelpad=8)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.02)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())


# ── Legend & title ────────────────────────────────────────────────────
ax.legend(loc="lower right", frameon=True, framealpha=0.9, edgecolor="gray")
#ax.set_title("ROC Curve — DNN Classifier", pad=10)

plt.tight_layout()
plt.savefig("roc_curve.pdf")
plt.savefig("roc_curve.png")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

# ===============================
# USER SETTINGS
# ===============================

dataset_path = "/content/drive/My Drive/DLData/TaMuDecay"

new_data_path = f"{dataset_path}/testdataWW0jwmvisv41mrun.txt"
new_datatt_path = f"{dataset_path}/testdatatt0jv51mrun.txt"

col_names = [
    "mu_px", "mu_py", "mu_pz",
    "e_px", "e_py", "e_pz",
    "nu_px", "nu_py",
    "mvis"
]

ww_df = pd.read_csv(new_data_path, sep=',', header=None)
ww_df.columns = col_names
ww_df = ww_df.apply(pd.to_numeric)

tt_df = pd.read_csv(new_datatt_path, sep=',', header=None)
tt_df.columns = col_names
tt_df = tt_df.apply(pd.to_numeric)

ww_df = recompute_mcol(ww_df)
tt_df = recompute_mcol(tt_df)

mass_points = [200, 250, 300, 350, 400, 450]
CUT_VALUES = np.arange(0.1, 0.99, 0.01)

L = 35.9      # fb^-1 (CHANGE if needed)
stt = 0.087   # pb (CHANGE if needed)
sww = 0.152    # pb (CHANGE if needed)
Ntt_gen = 1000000
Nww_gen = 1000000
Nsig_gen = 100000   # or correct value per mass

bins = [40, 80, 100, 125, 150, 175, 200, 235, 270, 315, 350,
        400, 450, 500, 560, 630, 700, 780, 860, 950, 1040, 1200]


# Predict WW
X_ww = scaler.transform(ww_df[feature_cols].values)
ww_probs = model.predict(X_ww).flatten()
ww_colmass = ww_df["mcol"].values

# Predict tt
X_tt = scaler.transform(tt_df[feature_cols].values)
tt_probs = model.predict(X_tt).flatten()
tt_colmass = tt_df["mcol"].values

# ===============================
# Likelihood function
# ===============================

def lchi2(b, s):
    if b <= 0:
        return 0
    return (b + s) - b * np.log((b + s) / b)

# ===============================
# LOOP OVER MASS POINTS
# ===============================

all_results = []
results_0j = {}

for mass in mass_points:

    print(f"\n=========== Mass {mass} GeV ===========")

    # Load independent signal file
    sig_path = f"{dataset_path}/testdatataulepm{mass}v2.txt"
    sig_df = pd.read_csv(sig_path, sep=',', header=None)
    sig_df.columns = col_names
    sig_df = sig_df.apply(pd.to_numeric)
    sig_df = recompute_mcol(sig_df)

    X_sig = scaler.transform(sig_df[feature_cols].values)
    sig_probs = model.predict(X_sig).flatten()
    sig_colmass = sig_df["mcol"].values

    best_xs = np.inf
    best_cut = None

    for CUT_VAL in CUT_VALUES:

        # Apply cut
        tt_mask = tt_probs > CUT_VAL
        ww_mask = ww_probs > CUT_VAL
        sig_mask = sig_probs > CUT_VAL

        tt_hist, _ = np.histogram(tt_colmass[tt_mask], bins=bins)
        ww_hist, _ = np.histogram(ww_colmass[ww_mask], bins=bins)
        sig_hist, _ = np.histogram(sig_colmass[sig_mask], bins=bins)

        # Scale background
        n_tt_scaled = stt * 1000 * L * tt_hist / Ntt_gen
        n_ww_scaled = sww * 1000 * L * ww_hist / Nww_gen

        n_background = n_tt_scaled + n_ww_scaled

        chi0 = np.sum([lchi2(b, 0) for b in n_background])

        def equation(xs):

            scaled_sig = xs * 1000 * L * sig_hist / Nsig_gen

            q = 0

            for i in range(len(sig_hist)):
              b = n_background[i]
              s = scaled_sig[i]

              if b > 0:
                  q += 2 * ( s - b * np.log(1 + s/b) )

            return q - 3.84

        try:
            xs_solution = brentq(equation, 1e-5, 100)

            if xs_solution < best_xs:
                best_xs = xs_solution
                best_cut = CUT_VAL

                # SAVE HISTOGRAMS FOR THIS BEST CUT
                best_sig_hist = sig_hist.copy()
                best_bkg_hist = n_background.copy()



        except:
            continue

    results_0j[mass] = {
        "sig_hist": best_sig_hist,
        "bkg_hist": best_bkg_hist,
        "best_cut": best_cut,
        "limit": best_xs
    }

    print(f"Best Cut: {best_cut}")
    print(f"95% CL Limit (pb): {best_xs:.5f}")

    all_results.append({
        "Mass": mass,
        "BestCut": best_cut,
        "Limit_pb": best_xs,
        "sig_hist": best_sig_hist,
        "bkg_hist": best_bkg_hist
    })



# ===============================
# Final Results
# ===============================

df_limits = pd.DataFrame(all_results)

print("\n=== Final Limit Results ===")
print(df_limits)


In [ ]:
import pickle

save_path = "/content/drive/My Drive/DLData/results_0j_newarch.pkl"

with open(save_path, "wb") as f:
    pickle.dump(results_0j, f)

print("0j results saved!")

In [ ]:
pip install shap

In [ ]:
import shap
import numpy as np

In [ ]:
# Create a wrapper function so SHAP only sees standard numpy arrays
f = lambda x: model.predict(x, verbose=0)

# Define X_train_np as X_train (already a numpy array)
X_train_np = X_train

# Define X_test_sample (taking a small sample for faster computation and visualization)
X_test_sample = shap.sample(X_test, 200)

# Define feature_names
feature_names = feature_cols

# Initialize KernelExplainer with background data (keep background small, e.g., 50)
explainer = shap.KernelExplainer(f, shap.sample(X_train_np, 50))

# Calculate SHAP values
shap_values = explainer.shap_values(X_test_sample, nsamples=200)

# The shap_values is a 3D array (num_samples, num_features, 1) for single output models.
# Squeeze the last dimension to make it 2D (num_samples, num_features) for plotting.
shap_values_to_plot = shap_values.squeeze()

# Plot
shap.summary_plot(
    shap_values_to_plot,
    X_test_sample,
    feature_names=feature_names,
    show=False   # IMPORTANT
)

plt.title("SHAP Summary Plot – NN + mcol Model (0j case)")
plt.tight_layout()
plt.show()

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# 1. JOURNAL-READY STYLING
# We decrease figsize so that when the image is placed in a LaTeX column,
# the text doesn't need to be scaled down (which makes it unreadable).
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
    "font.size": 10,                # Standard journal font size
    "axes.titlesize": 11,           # Slightly larger title
    "axes.labelsize": 10,           # Axis labels matching text
    "figure.dpi": 200,              # High resolution for screen work
    "savefig.dpi": 600,             # Production quality resolution
})

# 2. DEFINE THE LATEX MAPPING
latex_names = {
    "mcol": r"$M_{\mathrm{col}}$",
    "mvis": r"$m_{\mathrm{vis}}$",
    "nu_px": r"$\nu_{p_x}$",
    "nu_py": r"$\nu_{p_y}$",
    "e_px": r"$e_{p_x}$",
    "e_py": r"$e_{p_y}$",
    "e_pz": r"$e_{p_z}$",
    "mu_px": r"$\mu_{p_x}$",
    "mu_py": r"$\mu_{p_y}$",
    "mu_pz": r"$\mu_{p_z}$"
}

formatted_feature_names = [latex_names.get(col, col) for col in feature_cols]

# 3. SHAP COMPUTATION
f = lambda x: model.predict(x, verbose=0)
X_test_sample = shap.sample(X_test, 100)
explainer = shap.KernelExplainer(f, shap.sample(X_train, 50))
shap_values = explainer.shap_values(X_test_sample, nsamples=200) # Increased nsamples for stability
shap_values_to_plot = shap_values.squeeze()

# 4. PLOTTING (Optimized for Column Width)
# A standard single-column width is ~3.5 inches.
fig = plt.figure(figsize=(4.5, 3.5))

shap.summary_plot(
    shap_values_to_plot,
    X_test_sample,
    feature_names=formatted_feature_names,
    plot_type="dot",
    max_display=4,        # Only show top 4 to reduce "crowding"
    show=False,
    plot_size=None        # Prevents SHAP from overriding our figsize
)

# Manually adjust label sizes that SHAP sometimes hardcodes
#plt.title(r"SHAP Importance: NN + $M_{\mathrm{col}}$ Model (0j)", pad=15)
plt.xlabel("SHAP value (impact on model output)", labelpad=10)

# 5. FINAL TOUCHES FOR JOURNALS
plt.tight_layout()
# Saving as PDF ensures all text is vector-based (infinitely zoomable)
plt.savefig("shap_importance_physics_0j.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import rcParams

# ── Publication style ────────────────────────────────────────────────
rcParams.update({
    "font.family":         "serif",
    "font.serif":          ["Times New Roman", "DejaVu Serif"],
    "font.size":           13,
    "axes.titlesize":      14,
    "axes.labelsize":      13,
    "xtick.labelsize":     11,
    "ytick.labelsize":     11,
    "axes.linewidth":      1.2,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.top":           True,
    "ytick.right":         True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "figure.dpi":          150,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
})

# ── Data ─────────────────────────────────────────────────────────────
mcol_idx = feature_cols.index("mcol") # Define mcol_idx
X_test_sample_unscaled = scaler.inverse_transform(X_test_sample) # Unscale the data
mcol     = X_test_sample_unscaled[:, mcol_idx]
mvis_idx = feature_cols.index("mvis")
mvis     = X_test_sample_unscaled[:, mvis_idx]
pred     = model.predict(X_test_sample).flatten()

mask     = mcol < 500
mcol_cut, mvis_cut, pred_cut = mcol[mask], mvis[mask], pred[mask]

# ── Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    mcol_cut, mvis_cut,
    c=pred_cut,
    cmap="seismic",          # diverging: blue=background, red=signal
    alpha=0.6,
    s=12,                     # smaller points → less overlap
    linewidths=0,             # no marker edge for cleaner look
    vmin=0, vmax=1,           # fix colour scale to [0, 1]
    rasterized=True,          # flatten points in PDF (faster rendering)
)

# ── Colorbar ─────────────────────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("NN Classifier Score", labelpad=10)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.ax.tick_params(labelsize=10)

# ── Axes ─────────────────────────────────────────────────────────────
ax.set_xlabel(r"$M_\mathrm{col}$ [GeV]", labelpad=8)
ax.set_ylabel(r"$m_\mathrm{vis}$ [GeV]", labelpad=8)
ax.set_title(r"NN Score in $(M_\mathrm{col},\, m_\mathrm{vis})$ Plane"
             , pad=10)

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

# ── Save ─────────────────────────────────────────────────────────────
plt.tight_layout()
plt.savefig("nn_score_2d_1j.pdf")
plt.savefig("nn_score_2d_1j.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import rcParams
from scipy.stats import binned_statistic_2d

# ── Publication style specifically for the Binned Plot ───────────────
rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["DejaVu Serif"], # Reliable across systems
    "font.size":         14,               # Increased base size
    "axes.titlesize":    14,               # Clear title
    "axes.labelsize":    18,               # Large, readable axis labels
    "xtick.labelsize":   14,               # Readable numbers
    "ytick.labelsize":   12,
    "legend.fontsize":   12,
    "axes.linewidth":    1.5,              # Thicker spine for contrast
    "figure.figsize":    (7, 6),           # Keep aspect ratio
    "savefig.bbox":      "tight",
})

# ── Data Preparation (from previous cells and kernel state) ──────────
mcol_idx = feature_cols.index("mcol")
mvis_idx = feature_cols.index("mvis")

X_test_sample_unscaled = scaler.inverse_transform(X_test_sample)
mcol     = X_test_sample_unscaled[:, mcol_idx]
mvis     = X_test_sample_unscaled[:, mvis_idx]
pred     = model.predict(X_test_sample).flatten()

# Apply a mask consistent with the plot limits
mask     = (mcol >= 140) & (mcol <= 500) & (mvis >= 50) & (mvis <= 470)
mcol_cut, mvis_cut, pred_cut = mcol[mask], mvis[mask], pred[mask]

# Binning for the 2D histogram
x_bins = np.linspace(140, 500, 30) # Adjusted to match xlim
y_bins = np.linspace(50, 470, 30)  # Adjusted to match ylim

bin_means, x_edges, y_edges, _ = binned_statistic_2d(
    mcol_cut, mvis_cut, pred_cut, statistic='mean', bins=[x_bins, y_bins]
)


# Define x_trend for the overlay line
x_trend = np.linspace(140, 500, 100)

fig, ax = plt.subplots()

# Use shading='flat' for crisp blocks
im = ax.pcolormesh(x_edges, y_edges, bin_means.T,
                   cmap="seismic", vmin=0, vmax=1,
                   shading='flat', edgecolors='none')

# Trend line - make it thicker (lw=2.5) to stand out against the bins
ax.plot(x_trend, 0.7 * x_trend, color='black', linestyle='--',
        linewidth=2.5, label=r'$m_{\rm vis} = 0.7 \cdot M_{\rm col}$')

# ── Colorbar Refinement ──────────────────────────────────────────────
# Adding a smaller shrink factor can help the colorbar not dwarf the plot
cbar = fig.colorbar(im, ax=ax, pad=0.03, shrink=0.9)
cbar.set_label(r"Mean NN Score", labelpad=15, fontsize=14)

# ── Axis Formatting ──────────────────────────────────────────────────
ax.set_xlabel(r"$M_{\rm col}$ (GeV)", labelpad=12) # Added padding
ax.set_ylabel(r"$m_{\rm vis}$ (GeV)", labelpad=12)

# Ensure the limits match your data exactly to remove white space
ax.set_xlim(140, 500)
ax.set_ylim(50, 470)

plt.tight_layout()
plt.savefig("binned_nn_score_0j.pdf", dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_2d
from matplotlib import rcParams

# ── Update rcParams for the new plot ────────────────────────
rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.labelsize": 18,
    "figure.figsize": (7, 6),
})

# --- Fix: Generate data for plotting directly from data_B_msc to include msc consistently ---
# Assuming we want to plot signal events where msc is defined.
# Take a sample of signal events comparable to X_test_sample size.
num_samples_for_plot = len(X_test_sample) # or choose an appropriate number

# Ensure data_B_msc contains Label column if you intend to filter by it
# If data_B_msc always contains signal (Label=1) without mixing, then no filter needed.
# Otherwise, filter:
plot_data_signal = data_B_msc.sample(n=num_samples_for_plot, random_state=42)

# Extract msc for these sampled signal events
msc = plot_data_signal["msc"].values

# Extract features (including mvis) for prediction
features_for_pred = plot_data_signal[feature_cols].values

# Scale these features
scaled_features_for_pred = scaler.transform(features_for_pred)

# Get predictions for these signal events
pred = model.predict(scaled_features_for_pred).flatten()

# Extract mvis from the unscaled features_for_pred
mvis_idx = feature_cols.index("mvis")
mvis = features_for_pred[:, mvis_idx]
# ────────────────────────────────────────────────────────────────


# 3. Apply a mask based on the true mass range and visible mass range
# Adjusting limits: msc usually ranges from 200 to 500 in your dataset
mask = (msc >= 150) & (msc <= 500) & (mvis >= 50) & (mvis <= 500)
msc_cut, mvis_cut, pred_cut = msc[mask], mvis[mask], pred[mask]

# 4. Define Bins for msc vs mvis
x_bins = np.linspace(150, 500, 30) # X-axis is now msc
y_bins = np.linspace(50, 500, 30)  # Y-axis is mvis

bin_means, x_edges, y_edges, _ = binned_statistic_2d(
    msc_cut, mvis_cut, pred_cut, statistic='mean', bins=[x_bins, y_bins]
)

# ── Plotting ─────────────────────────────────────────────────
fig, ax = plt.subplots()

im = ax.pcolormesh(x_edges, y_edges, bin_means.T,
                   cmap="seismic", vmin=0, vmax=1,
                   shading='flat', edgecolors='none')

# Trend line for True Mass: mvis is typically ~0.6-0.7 of true mass
x_trend = np.linspace(150, 500, 100)
ax.plot(x_trend, 0.7 * x_trend, color='black', linestyle='--',
        linewidth=2.5,)

# Labels and Colorbar
cbar = fig.colorbar(im, ax=ax, pad=0.03, shrink=0.9)
cbar.set_label(r"Mean NN Score", labelpad=15)

ax.set_xlabel(r"$m_{H}$ (GeV)", labelpad=12)
ax.set_ylabel(r"$m_{\rm vis}$ (GeV)", labelpad=12)

ax.set_xlim(150, 500)
ax.set_ylim(50, 500)


plt.tight_layout()
plt.savefig("binned_nn_score_msc.pdf", dpi=300)
plt.show()